# Pipeline Unificado de Laboratorios (Labs 4, 5, 6, 7 y 8)
## Generación de `data/raw/finanzas_igae_mensual_final.parquet` (2003 - 2026)

Este notebook consolida exclusivamente los laboratorios base del curso:
1. **Finanzas Públicas (Labs 4 y 5):** Ingesta, unión histórica (1990-2026) y filtro de series `XBB` (Ingresos) y `XBC` (Gastos).
2. **Deflactación (Lab 6):** Ajuste a precios constantes de diciembre de 2025 usando el INPC.
3. **Actividad Económica (Labs 7 y 8):** Ingesta de `igae_limpio.parquet` y pivote sectorial.
4. **Integración:** Cruce mensual 1:1 de Finanzas e IGAE sin incluir S&P 500 ni Tipo de Cambio (esos se unifican en `pipeline.py`).

In [1]:
import os
import sys
import shutil
from pathlib import Path

# ==============================================================================
# 1. CONFIGURACIÓN DE PARÁMETROS Y RUTAS DINÁMICAS
# ==============================================================================
# Ventana temporal configurada desde 2003 (cobertura completa de INPC)
FECHA_INICIO = "2003-01-01"
FECHA_FIN    = "2026-07-01"

# Detección automática de la raíz del proyecto (funciona dentro o fuera de notebooks/)
RUTA_BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RUTA_SHCP_HIST = RUTA_BASE / "data" / "raw" / "ingreso_gasto_historico.csv"
RUTA_SHCP_ACT  = RUTA_BASE / "data" / "raw" / "ingreso_gasto.csv"
RUTA_INPC      = RUTA_BASE / "data" / "raw" / "conjunto_de_datos_inpc_mensual.csv"
RUTA_IGAE      = RUTA_BASE / "data" / "raw" / "igae_limpio.parquet"

# Salida final que espera el pipeline modular original
RUTA_SALIDA_PARQUET = RUTA_BASE / "data" / "raw" / "finanzas_igae_mensual_final.parquet"

# Periodo base para deflactación (Lab 6)
ANIO_BASE = 2025
MES_BASE  = 12

print("Ruta base del proyecto:", RUTA_BASE.resolve())
print("Ruta de salida configurada:", RUTA_SALIDA_PARQUET.resolve())
print("¿Existe SHCP Histórico?: ", RUTA_SHCP_HIST.exists())
print("¿Existe SHCP Actual?:    ", RUTA_SHCP_ACT.exists())
print("¿Existe INPC?:           ", RUTA_INPC.exists())
print("¿Existe IGAE Parquet?:   ", RUTA_IGAE.exists())

Ruta base del proyecto: C:\Users\julia\OneDrive\Escritorio\MCD\Procesamiento de GBD\Proyecto_Integrador_PGBD
Ruta de salida configurada: C:\Users\julia\OneDrive\Escritorio\MCD\Procesamiento de GBD\Proyecto_Integrador_PGBD\data\raw\finanzas_igae_mensual_final.parquet
¿Existe SHCP Histórico?:  True
¿Existe SHCP Actual?:     True
¿Existe INPC?:            True
¿Existe IGAE Parquet?:    True


In [3]:
# ==============================================================================
# 2. INICIALIZACIÓN DE SPARK (Compatible con Windows y Anaconda)
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .appName("Pipeline_Laboratorios_Finanzas_IGAE")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"[OK] Sesión de Spark lista: {spark.version}")

[OK] Sesión de Spark lista: 3.5.9


In [4]:
# ==============================================================================
# 3. FINANZAS PÚBLICAS (SHCP - Labs 4 y 5)
# ==============================================================================
print("[SHCP] Cargando fuentes histórica y contemporánea...")

df_act = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(str(RUTA_SHCP_ACT))
)

if RUTA_SHCP_HIST.exists():
    df_hist = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .csv(str(RUTA_SHCP_HIST))
    )
    df_shcp_raw = df_act.unionByName(df_hist)
else:
    print("[AVISO] Archivo histórico no encontrado, usando solo contemporáneo.")
    df_shcp_raw = df_act

# Filtrar exactamente las series XBB (Ingreso) y XBC (Gasto Total) del Gobierno Federal
# y eliminar duplicados como se hizo en el Laboratorio 5
df_shcp_filtrado = (
    df_shcp_raw
    .filter(F.col("CLAVE_DE_CONCEPTO").isin(["XBB", "XBC"]))
    .dropDuplicates(["CICLO", "MES", "CLAVE_DE_CONCEPTO"])
)

# Mapeo numérico de meses para sincronización
meses_dict = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4, "Mayo": 5, "Junio": 6,
    "Julio": 7, "Agosto": 8, "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12
}
map_expr = F.create_map([F.lit(x) for i in meses_dict.items() for x in i])

df_finanzas = (
    df_shcp_filtrado
    .withColumn("MES_NUM", map_expr[F.col("MES")])
    .withColumn("MONTO", F.col("MONTO").cast("double"))
    .withColumn("CICLO", F.col("CICLO").cast("integer"))
    .withColumn("FECHA", F.to_date(F.concat_ws("-", F.col("CICLO"), F.lpad(F.col("MES_NUM"), 2, "0"), F.lit("01"))))
)

print(f"[SHCP] Registros filtrados de Ingresos y Gastos: {df_finanzas.count()}")
df_finanzas.select("FECHA", "CICLO", "MES", "CLAVE_DE_CONCEPTO", "MONTO").show(5, truncate=False)

[SHCP] Cargando fuentes histórica y contemporánea...
[SHCP] Registros filtrados de Ingresos y Gastos: 878
+----------+-----+---------+-----------------+-----------+
|FECHA     |CICLO|MES      |CLAVE_DE_CONCEPTO|MONTO      |
+----------+-----+---------+-----------------+-----------+
|1990-04-01|1990 |Abril    |XBB              |9702308.0  |
|1990-04-01|1990 |Abril    |XBC              |1.0043878E7|
|1990-08-01|1990 |Agosto   |XBB              |9737233.0  |
|1990-08-01|1990 |Agosto   |XBC              |9950257.0  |
|1990-12-01|1990 |Diciembre|XBB              |1.3107627E7|
+----------+-----+---------+-----------------+-----------+
only showing top 5 rows



In [5]:
# ==============================================================================
# 4. INPC Y DEFLACTACIÓN A PRECIOS CONSTANTES (Lab 6)
# ==============================================================================
print("[INPC] Cargando y filtrando índice de precios general...")

df_inpc_raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(str(RUTA_INPC))
)

# Filtrar serie general del INPC y calcular llaves temporales
df_inpc = (
    df_inpc_raw
    .filter(F.lower(F.col("CONCEPTO")).contains("precios al consumidor (inpc)"))
    .withColumn("FECHA_INPC", F.to_date(F.col("FECHA")))
    .withColumn("CICLO", F.year(F.col("FECHA_INPC")))
    .withColumn("MES_NUM", F.month(F.col("FECHA_INPC")))
    .withColumn("INPC", F.col("VALOR").cast("double"))
    .select("CICLO", "MES_NUM", "INPC")
    .dropDuplicates(["CICLO", "MES_NUM"])
)

# Obtener valor base (diciembre 2025)
inpc_base_row = df_inpc.filter((F.col("CICLO") == ANIO_BASE) & (F.col("MES_NUM") == MES_BASE)).first()
if inpc_base_row:
    INPC_BASE = float(inpc_base_row["INPC"])
else:
    # Valor documentado en Lab 6
    INPC_BASE = 143.042
print(f"[INPC] Periodo Base ({ANIO_BASE}-{MES_BASE}): INPC_BASE = {INPC_BASE}")

# Cruce con Finanzas y cálculo de montos reales
df_finanzas_real = (
    df_finanzas
    .join(df_inpc, on=["CICLO", "MES_NUM"], how="inner")
    .withColumn("MONTO_REAL", F.col("MONTO") * (F.lit(INPC_BASE) / F.col("INPC")))
)

# Agregación mensual de Ingresos y Gastos Reales (Lab 8)
df_finanzas_mensual = (
    df_finanzas_real
    .withColumn("PERIODO", F.trunc(F.col("FECHA"), "month"))
    .groupBy("PERIODO")
    .agg(
        F.first(F.when(F.col("CLAVE_DE_CONCEPTO") == "XBB", F.col("MONTO_REAL")), ignorenulls=True).alias("INGRESOS_REALES"),
        F.first(F.when(F.col("CLAVE_DE_CONCEPTO") == "XBC", F.col("MONTO_REAL")), ignorenulls=True).alias("GASTOS_REALES"),
        F.first("INPC", ignorenulls=True).alias("INPC")
    )
    .orderBy("PERIODO")
)

print(f"[FINANZAS] Meses reales agregados: {df_finanzas_mensual.count()}")
df_finanzas_mensual.show(5, truncate=False)

[INPC] Cargando y filtrando índice de precios general...
[INPC] Periodo Base (2025-12): INPC_BASE = 143.042
[FINANZAS] Meses reales agregados: 283
+----------+--------------------+--------------------+---------------+
|PERIODO   |INGRESOS_REALES     |GASTOS_REALES       |INPC           |
+----------+--------------------+--------------------+---------------+
|2003-01-01|3.133722835712268E8 |2.83901609001714E8  |53.525440675316|
|2003-02-01|2.2651803238740107E8|2.588787552467126E8 |53.674122454969|
|2003-03-01|2.7395978697662276E8|2.5481907005789644E8|54.012930412786|
|2003-04-01|2.9986526379735494E8|2.3905445272123864E8|54.10514419947 |
|2003-05-01|2.2972073924542564E8|2.4208449779621354E8|53.930559670749|
+----------+--------------------+--------------------+---------------+
only showing top 5 rows



In [6]:
# ==============================================================================
# 5. IGAE HISTÓRICO Y PIVOTE SECTORIAL (Labs 7 y 8)
# ==============================================================================
print(f"[IGAE] Leyendo serie histórica desde: {RUTA_IGAE}")

if RUTA_IGAE.suffix == ".parquet":
    df_igae_raw = spark.read.parquet(str(RUTA_IGAE))
else:
    df_igae_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RUTA_IGAE))

cols_igae = set(df_igae_raw.columns)

# Si ya viene pivoteado
if {"Total", "Actividades primarias", "Actividades secundarias", "Actividades terciarias"}.issubset(cols_igae):
    col_fecha = "PERIODO" if "PERIODO" in cols_igae else "FECHA"
    df_igae_mensual = (
        df_igae_raw
        .withColumn("PERIODO", F.trunc(F.to_date(F.col(col_fecha)), "month"))
        .select("PERIODO", "Total", "Actividades primarias", "Actividades secundarias", "Actividades terciarias")
        .dropDuplicates(["PERIODO"])
        .orderBy("PERIODO")
    )
else:
    # Formato largo estándar de Lab 7
    df_igae_mensual = (
        df_igae_raw
        .withColumn("PERIODO", F.trunc(F.to_date(F.col("FECHA")), "month"))
        .groupBy("PERIODO")
        .pivot("ACTIVIDAD", ["Total", "Actividades primarias", "Actividades secundarias", "Actividades terciarias"])
        .agg(F.first("IGAE", ignorenulls=True))
        .orderBy("PERIODO")
    )

print(f"[IGAE] Meses únicos de IGAE procesados: {df_igae_mensual.count()}")
df_igae_mensual.show(5, truncate=False)

[IGAE] Leyendo serie histórica desde: c:\Users\julia\OneDrive\Escritorio\MCD\Procesamiento de GBD\Proyecto_Integrador_PGBD\data\raw\igae_limpio.parquet
[IGAE] Meses únicos de IGAE procesados: 403
+----------+----------------+---------------------+-----------------------+----------------------+
|PERIODO   |Total           |Actividades primarias|Actividades secundarias|Actividades terciarias|
+----------+----------------+---------------------+-----------------------+----------------------+
|1993-01-01|55.4347362001522|58.7961983277473     |63.3077816049447       |50.7752581279791      |
|1993-02-01|56.4569707543221|60.7638163380742     |64.8891300827476       |51.427224440806       |
|1993-03-01|58.900549489378 |55.9649527854709     |68.0704745065654       |53.8552136813162      |
|1993-04-01|57.1358437612133|58.9891639211639     |64.5246144916617       |52.8354746449278      |
|1993-05-01|57.8918526898321|61.7620784097257     |64.3828078068416       |53.9891128823904      |
+----------+

In [7]:
# ==============================================================================
# 6. INTEGRACIÓN MENSUAL 1:1 Y FILTRO DE VENTANA TEMPORAL (Lab 8)
# ==============================================================================
print("[JOIN] Cruzando Finanzas Reales + IGAE por PERIODO...")

df_lab8_final = (
    df_finanzas_mensual
    .join(df_igae_mensual, on="PERIODO", how="inner")
    .filter(F.col("PERIODO").between(FECHA_INICIO, FECHA_FIN))
    .select(
        "PERIODO",
        "INGRESOS_REALES",
        "GASTOS_REALES",
        "INPC",
        "Total",
        "Actividades primarias",
        "Actividades secundarias",
        "Actividades terciarias"
    )
    .orderBy("PERIODO")
)

total_registros = df_lab8_final.count()
print(f"\n[RESULTADO] Total de meses integrados ({FECHA_INICIO} a {FECHA_FIN}): {total_registros}")
df_lab8_final.show(10, truncate=False)

[JOIN] Cruzando Finanzas Reales + IGAE por PERIODO...

[RESULTADO] Total de meses integrados (2003-01-01 a 2026-07-01): 283
+----------+--------------------+--------------------+---------------+----------------+---------------------+-----------------------+----------------------+
|PERIODO   |INGRESOS_REALES     |GASTOS_REALES       |INPC           |Total           |Actividades primarias|Actividades secundarias|Actividades terciarias|
+----------+--------------------+--------------------+---------------+----------------+---------------------+-----------------------+----------------------+
|2003-01-01|3.133722835712268E8 |2.83901609001714E8  |53.525440675316|72.5708002109464|78.7727273038742     |80.8620385334646       |67.5155238917463      |
|2003-02-01|2.2651803238740107E8|2.588787552467126E8 |53.674122454969|70.9412460357379|67.5871743790872     |80.4523791763127       |65.7254044557108      |
|2003-03-01|2.7395978697662276E8|2.5481907005789644E8|54.012930412786|73.5222964863106|66.4

In [8]:
# ==============================================================================
# 7. AUDITORÍA DE CALIDAD Y CONTRATOS (0 Nulos y Unicidad)
# ==============================================================================
duplicados = df_lab8_final.groupBy("PERIODO").count().filter(F.col("count") > 1).count()
assert duplicados == 0, f"[ERROR] Se detectaron {duplicados} meses duplicados."

nulos = df_lab8_final.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_lab8_final.columns]).first().asDict()
print("Conteo de nulos por columna:", nulos)
assert sum(nulos.values()) == 0, "[ERROR] Existen valores nulos en el dataset."

print("[OK] Validación superada: 0 duplicados y 0 valores nulos.")

Conteo de nulos por columna: {'PERIODO': 0, 'INGRESOS_REALES': 0, 'GASTOS_REALES': 0, 'INPC': 0, 'Total': 0, 'Actividades primarias': 0, 'Actividades secundarias': 0, 'Actividades terciarias': 0}
[OK] Validación superada: 0 duplicados y 0 valores nulos.


In [9]:
# ==============================================================================
# 8. PERSISTENCIA EN data/raw/finanzas_igae_mensual_final.parquet
# (Solución protegida contra WinError 5 de Windows)
# ==============================================================================
import pandas as pd

print(f"[LOAD] Guardando resultado en: {RUTA_SALIDA_PARQUET}")
os.makedirs(RUTA_SALIDA_PARQUET.parent, exist_ok=True)

# Limpiar versión previa si existe para evitar conflictos de bloqueo de archivos en Windows
if RUTA_SALIDA_PARQUET.exists():
    if RUTA_SALIDA_PARQUET.is_dir():
        shutil.rmtree(RUTA_SALIDA_PARQUET)
    else:
        RUTA_SALIDA_PARQUET.unlink()

# Guardar con PyArrow
df_pd = df_lab8_final.toPandas()
df_pd.to_parquet(str(RUTA_SALIDA_PARQUET), index=False, engine="pyarrow")
print("[OK] Archivo generado exitosamente.")

# Verificación de relectura con Spark
verif = spark.read.parquet(str(RUTA_SALIDA_PARQUET))
print(f"Verificación física: {verif.count()} filas y {len(verif.columns)} columnas.")
verif.show(5, truncate=False)

[LOAD] Guardando resultado en: c:\Users\julia\OneDrive\Escritorio\MCD\Procesamiento de GBD\Proyecto_Integrador_PGBD\data\raw\finanzas_igae_mensual_final.parquet
[OK] Archivo generado exitosamente.
Verificación física: 283 filas y 8 columnas.
+----------+--------------------+--------------------+---------------+----------------+---------------------+-----------------------+----------------------+
|PERIODO   |INGRESOS_REALES     |GASTOS_REALES       |INPC           |Total           |Actividades primarias|Actividades secundarias|Actividades terciarias|
+----------+--------------------+--------------------+---------------+----------------+---------------------+-----------------------+----------------------+
|2003-01-01|3.133722835712268E8 |2.83901609001714E8  |53.525440675316|72.5708002109464|78.7727273038742     |80.8620385334646       |67.5155238917463      |
|2003-02-01|2.2651803238740107E8|2.588787552467126E8 |53.674122454969|70.9412460357379|67.5871743790872     |80.4523791763127     